In [1]:
# To calculate log-like we will use the original n(z), Cell's calculated using this n(z), and the covariance matrix #

# General imports #
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
from scipy.interpolate import interp1d
from scipy import integrate
from copy import deepcopy

# Import euclidlib for reading the Euclid data #
import euclidlib as el

# Import cloelib for cosmology and theoretical predictions #
from cloelib.cosmology.camb_cosmology import CAMBBackground
from cloelib.cosmology.HMcode2020Emu_cosmology import HMemuLinearPerturbations, HMemuNonLinearPerturbations
from cloelib.observables.photo import ShearTracer, PositionsTracer
from cloelib.summary_statistics.angular_two_point import AngularTwoPoint

# Import cloelike for likelihood computations #
from cloelike.EuclidLikelihood_photo_Cls import EuclidLikelihood_3x2pt




2025-11-24 13:13:02.745785: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


/home/jipdebuck/miniconda3/envs/venv_310/lib/python3.10/site-packages/HMcode2020Emu
Loading linear emulator...


2025-11-24 13:13:03.666429: E tensorflow/compiler/xla/stream_executor/cuda/cuda_driver.cc:268] failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected


Linear emulator loaded in memory.
Loading nonlinear emulator...
Non-linear emulator loaded in memory.
Loading linear emulator...
Baryonic boost emulator loaded in memory.
Loading sigma8 emulator...
Linear emulator loaded in memory.


In [2]:
# Import and prepare original n(z) distribution #

zs = np.linspace(1e-4, 3, 100)
z, nz = el.photo.redshift_distributions('/home/jipdebuck/nzTabSPV3.fits')

def normalize_dndz(nz_example, z_nz):
    normalized = {}
    for key in nz_example:
        normalized[key] = nz_example[key] / integrate.trapezoid(nz_example[key], z_nz)
    return normalized

# Normalize both dndz_pos and dndz_she
dndz_pos_norm = normalize_dndz(nz, z)
dndz_she_norm = normalize_dndz(nz, z)

# Function to resample the normalized dndz
def resample_dndz(nz_example, z_nz, myz):
    my_dndz = np.vstack(list(nz_example.values())) #stacking arrays vertically
    my_dndz_norm = np.zeros([len(nz_example), len(myz)]) #creating an empty array
    for i in range(len(nz_example)):
        my_dndz_norm[i, :] = np.interp(myz, z_nz, my_dndz[i, :]) #interpolating to new myz values
    return my_dndz_norm

# Resampling the normalized dndz with 100 values
my_dndz_pos_norm = resample_dndz(dndz_pos_norm, z, zs)
my_dndz_she_norm = resample_dndz(dndz_she_norm, z, zs)

In [8]:
# Compute the C_ell's using the original n(z) and fiducial values from overview paper #

# The arguments of the Background functions follow the cosmology.API
background = CAMBBackground(H0=67.37, 
                            Omega_cdm0=0.27, 
                            Omega_b0=0.05, 
                            w0=-1, 
                            wa=0, 
                            Omega_k0 = 0.0, 
                            ns = 0.966, 
                            As = 2.09e-9,
                            mnu = 0.06,
                            gamma_MG = 0.545,
                            N_mnu = 1)
linear_perturbations_emu = HMemuLinearPerturbations(background, zs)
nonlinear_perturbations_emu = HMemuNonLinearPerturbations(background, linear_perturbations_emu, zs, log10TAGN=7.75)

# Select the perturbations object you want to use
perturbations = nonlinear_perturbations_emu

# Define the tracers
tracer_pos = PositionsTracer(perturbations=perturbations, 
                             dndz=my_dndz_pos_norm,
                             z = zs,
                             galaxy_bias_model='poly',
                             nuisance_params={'b1_photo_poly0': 1.33291, 'b1_photo_poly1': -0.72414,
                                              'b1_photo_poly2': 1.0183, 'b1_photo_poly3': -0.14913,
                                              'magnification_bias_1': 0.0, 'magnification_bias_2': 0.0,
                                              'magnification_bias_3': 0.0, 'magnification_bias_4': 0.0,
                                              'magnification_bias_5': 0.0, 'magnification_bias_6': 0.0,
                                              'magnification_bias_7': 0.0, 'magnification_bias_8': 0.0,
                                              'magnification_bias_9': 0.0, 'magnification_bias_10': 0.0,
                                              'magnification_bias_11': 0.0, 'magnification_bias_12': 0.0,
                                              'magnification_bias_13': 0.0,
                                              'dz_pos_1': -0.025749, 'dz_pos_2': 0.022716,
                                              'dz_pos_3': -0.026032, 'dz_pos_4': 0.012594,
                                              'dz_pos_5': 0.019285, 'dz_pos_6': 0.008326,
                                              'dz_pos_7': 0.038207, 'dz_pos_8': 0.002732,
                                              'dz_pos_9': 0.034066, 'dz_pos_10': 0.049479,
                                              'dz_pos_11': 0.066490, 'dz_pos_12': 0.000815,
                                              'dz_pos_13': 0.049070})

tracer_she = ShearTracer(perturbations=perturbations, 
                             dndz=my_dndz_she_norm,
                             z = zs,
                             nuisance_params={'AIA': 0.16, 'CIA': 0.0134, 'EtaIA':1.66,
                                              'multiplicative_bias_1': 0.00, 'multiplicative_bias_2': 0.00,
                                              'multiplicative_bias_3': 0.000, 'multiplicative_bias_4': 0.00,
                                              'multiplicative_bias_5': 0.00, 'multiplicative_bias_6': 0.00,
                                              'multiplicative_bias_7': 0.00, 'multiplicative_bias_8': 0.00,
                                              'multiplicative_bias_9': 0.00, 'multiplicative_bias_10': 0.00,
                                              'multiplicative_bias_11': 0.00, 'multiplicative_bias_12': 0.00,
                                              'multiplicative_bias_13': 0.00,
                                              'dz_shear_1': -0.025749, 'dz_shear_2': 0.022716,
                                              'dz_shear_3': -0.026032, 'dz_shear_4': 0.012594,
                                              'dz_shear_5': 0.019285, 'dz_shear_6': 0.008326,
                                              'dz_shear_7': 0.038207, 'dz_shear_8': 0.002732,
                                              'dz_shear_9': 0.034066, 'dz_shear_10': 0.049479,
                                              'dz_shear_11': 0.066490, 'dz_shear_12': 0.000815,
                                              'dz_shear_13': 0.049070})

In [9]:
# Compute angular two-point statistics #

nl = 100
ells = np.logspace(1., np.log10(3000), nl)

twopoint_pospos = AngularTwoPoint(tracer_pos, tracer_pos)
twopoint_shepos = AngularTwoPoint(tracer_she, tracer_pos)
twopoint_posshe = AngularTwoPoint(tracer_pos, tracer_she)
twopoint_sheshe = AngularTwoPoint(tracer_she, tracer_she)

In [5]:
data_downloaded = False

if data_downloaded == False:
    
    import requests
    
    # URLs of the files to be downloaded
    urls = {
        'nz_example.fits': 'https://zenodo.org/records/15092862/files/nz_example.fits',
        'mixmats_example.fits': 'https://zenodo.org/records/17191365/files/example-mixmats.fits',
        'cls_example.fits': 'https://zenodo.org/records/17191365/files/example-spectra.fits',
        'cov_WL.npz': 'https://zenodo.org/records/17191365/files/cov_WL.npz',
        'cov_GCph.npz': 'https://zenodo.org/records/17191365/files/cov_GC.npz',
        'cov_3x2pt.npz': 'https://zenodo.org/records/17191365/files/cov_3x2pt.npz'
    }
    
    
    # Function to download a file
    def download_file(url, filename):
        response = requests.get(url)
        if response.status_code == 200:
            with open(filename, 'wb') as f:
                f.write(response.content)
            print(f'{filename} downloaded successfully')
        else:
            print(f'Failed to download {filename}. Status code: {response.status_code}')
    
    # Download all files
    for filename, url in urls.items():
        download_file(url, filename)

nz_example.fits downloaded successfully
mixmats_example.fits downloaded successfully
cls_example.fits downloaded successfully
cov_WL.npz downloaded successfully
cov_GCph.npz downloaded successfully
cov_3x2pt.npz downloaded successfully


In [10]:
covmat_3x2pt = np.load("/home/jipdebuck/CovMat-3x2pt-GaussSSC-32Bins-13245deg2.npy")
mixmats = el.photo.mixing_matrices('mixmats_example.fits')
cls_data ={**twopoint_sheshe.get_Cl(ells, 0, perturbations.k)
, **twopoint_posshe.get_Cl(ells, 0, perturbations.k)
, **twopoint_pospos.get_Cl(ells, 0, perturbations.k)}

In [15]:
def build_data(ell_key, cov, include_pos=False, include_she=False):
    data = {
        'cells': cls_data,
        'ells': cls_data[ell_key].ell,
        'z_arr': zs,
        'cov': cov,
        'mixmat': mixmats,
    }
    if include_pos:
        data['dndz_pos'] = my_dndz_pos_norm
    if include_she:
        data['dndz_she'] = my_dndz_she_norm
    return data

def build_settings():
    scale_cuts = {key: [5, 3000] for key in cls_data}
    return {'n_ell_bins': 32, 'scale_cuts': scale_cuts}

# Build each dataset with correct dndz components
data_3x2pt  = build_data(('POS', 'POS', 1, 1), covmat_3x2pt,      include_pos=True, include_she=True)

# Settings (same for all, built from cells_data structure)
settings_3x2pt  = build_settings()

In [17]:
import time

likelihood_classes = {
    '3x2pt': EuclidLikelihood_3x2pt,
}

data_dicts = {
    '3x2pt': (data_3x2pt, settings_3x2pt),
}

like_tests = {}
loglike_results = {}

print("🔍 Timing log-likelihood initialisation (this only happens once) and displaying results:")

for label in ['3x2pt']:
    print(f"\n📊 {label} loglike:")
    data, settings = data_dicts[label]
    cls = likelihood_classes[label]

    start = time.time()
    like_tests[label] = cls(
        data=data,
        settings=settings,
        Background=CAMBBackground,
        LinPerturbations=HMemuLinearPerturbations,
        NonLinPerturbations=HMemuNonLinearPerturbations,
        mode = 'uncoupled'
    )
    end = time.time()
    print(f"⏱️ Time elapsed ({label}): {end - start:.3f} seconds")

🔍 Timing log-likelihood initialisation (this only happens once) and displaying results:

📊 3x2pt loglike:
⏱️ Time elapsed (3x2pt): 0.265 seconds


In [18]:
default_pars = {'H0':67.37,'Omega_cdm0':0.27,'Omega_b0':0.05,'ns':0.966,'As':2.09e-9,
                'w0':-1,'wa':0, 'Omega_k0':0,'mnu':0.06,'gamma_MG':0.545, 'N_mnu': 1, 
                'log10TAGN': 7.75,
                'AIA': 0.16, 'CIA': 0.0134, 'EtaIA':1.66,
                'b1_photo_poly0': 1.33291, 'b1_photo_poly1': -0.72414,
                'b1_photo_poly2': 1.0183, 'b1_photo_poly3': -0.14913,
                'magnification_bias_1': 0.0, 'magnification_bias_2': 0.0,
                'magnification_bias_3': 0.0, 'magnification_bias_4': 0.0,
                'magnification_bias_5': 0.0, 'magnification_bias_6': 0.0,
                'magnification_bias_7': 0.0, 'magnification_bias_8': 0.0,
                'magnification_bias_9': 0.0, 'magnification_bias_10': 0.0,
                'magnification_bias_11': 0.0, 'magnification_bias_12': 0.0,
                'magnification_bias_13': 0.0,
                'dz_pos_1': -0.025749, 'dz_pos_2': 0.022716,
                'dz_pos_3': -0.026032, 'dz_pos_4': 0.012594,
                'dz_pos_5': 0.019285, 'dz_pos_6': 0.008326,
                'dz_pos_7': 0.038207, 'dz_pos_8': 0.002732,
                'dz_pos_9': 0.034066, 'dz_pos_10': 0.049479,
                'dz_pos_11': 0.066490, 'dz_pos_12': 0.000815,
                'dz_pos_13': 0.049070,
                'multiplicative_bias_1': 0.00, 'multiplicative_bias_2': 0.00,
                'multiplicative_bias_3': 0.000, 'multiplicative_bias_4': 0.00,
                'multiplicative_bias_5': 0.00, 'multiplicative_bias_6': 0.00,
                'multiplicative_bias_7': 0.00, 'multiplicative_bias_8': 0.00,
                'multiplicative_bias_9': 0.00, 'multiplicative_bias_10': 0.00,
                'multiplicative_bias_11': 0.00, 'multiplicative_bias_12': 0.00,
                'multiplicative_bias_13': 0.00,
                'dz_shear_1': -0.025749, 'dz_shear_2': 0.022716,
                'dz_shear_3': -0.026032, 'dz_shear_4': 0.012594,
                'dz_shear_5': 0.019285, 'dz_shear_6': 0.008326,
                'dz_shear_7': 0.038207, 'dz_shear_8': 0.002732,
                'dz_shear_9': 0.034066, 'dz_shear_10': 0.049479,
                'dz_shear_11': 0.066490, 'dz_shear_12': 0.000815,
                 'dz_shear_13': 0.049070}

In [19]:
# Dictionary of pretty names for output
labels_pretty = {
    '3x2pt': "3x2pt"
}

print("⏱️ Timing log-likelihood evaluations and displaying results:")

loglike_results = {}

for label in ['3x2pt']:
    print(f"\n📊 {labels_pretty[label]} loglike:")
    start = time.time()
    result = like_tests[label].loglike(default_pars)
    elapsed = time.time() - start
    loglike_results[label] = result
    print(f"✅ Returned value ({label}): {result}")
    print(f"⏱️ Time elapsed ({label}): {elapsed:.3f} seconds")


⏱️ Timing log-likelihood evaluations and displaying results:

📊 3x2pt loglike:
✅ Returned value (3x2pt): -536709.6493490585
⏱️ Time elapsed (3x2pt): 41.143 seconds
